# P5_scores_features — selección robusta con validación cruzada repetida

Esta versión sustituye la validación fija de 7 casos por **RepeatedStratifiedKFold 5×20** sobre todo el conjunto de desarrollo 2021–2022 (train+val). El conjunto temporal 2023–2024 queda reservado para evaluación final y no interviene en la selección de variables.

**Nota metodológica:** las repeticiones de CV estabilizan la selección de variables, pero no constituyen las corridas independientes que se usarán después para la inferencia estadística del artículo.


In [3]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, average_precision_score
from sklearn.inspection import permutation_importance
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC

warnings.filterwarnings('ignore')

# ============================================================
# P5_scores_features - temporal + validacion cruzada repetida
# ============================================================
# Objetivo:
# 1) Unir train+val de 2021-2022 como conjunto de DESARROLLO.
# 2) Seleccionar/estimar importancia de variables SOLO dentro de desarrollo.
# 3) Usar RepeatedStratifiedKFold 5x20 para evitar depender de una validacion fija de 7 casos.
# 4) Mantener 2023-2024 completamente reservado hasta RUN_FINAL_TEST=True.
#
# Importante: las repeticiones de CV estabilizan la seleccion de variables,
# pero NO sustituyen los experimentos con multiples semillas independientes
# que se usaran despues para la inferencia estadistica del articulo.

# ============================================================
# CONFIGURACION
# ============================================================
BASE_DIR = Path(r"C:/Users/Dra. Pilar/UltimaversionProgramasComparacion")
SEED = 41
TOP_K = 7
N_SPLITS = 5
N_REPEATS_CV = 20
N_REPEATS_IMPORTANCE = 20
RUN_FINAL_TEST = False  # Mantener False hasta congelar Top-K

TRAIN_FILE = BASE_DIR / 'sample_temporal_train_hist2000_2020_target2021_2022_thr1_neg1_random_isolates.csv'
VAL_FILE   = BASE_DIR / 'sample_temporal_val_hist2000_2020_target2021_2022_thr1_neg1_random_isolates.csv'
TEST_FILE  = BASE_DIR / 'sample_temporal_test_hist2000_2022_target2023_2024_thr1_neg1_random_isolates.csv'

BASE_FEATURES = [
    'sum_of_papers',
    'sum_of_neighbors',
    'log_secundary_neighbors',
    'sum_of_keywords',
    'keywords_match',
    'sum_of_areas',
    'lenght_short_path',
    'clustering_index_sum',
    'jaccard_coefficient',
    'resource_allocation',
    'adamic_adar_index',
    'preferential_attachment',
    'community_cn',
    'community_ra',
]

# ============================================================
# CARGA
# ============================================================
def load_split(path):
    df = pd.read_csv(path)
    unnamed = [c for c in df.columns if c.startswith('Unnamed:')]
    if unnamed:
        df = df.drop(columns=unnamed)
    return df

train = load_split(TRAIN_FILE)
val = load_split(VAL_FILE)

required = {'source', 'target', 'connected', *BASE_FEATURES}
for name, df in [('train', train), ('val', val)]:
    missing = required.difference(df.columns)
    if missing:
        raise ValueError(f'{name}: faltan columnas: {sorted(missing)}')
    print(f'{name}: n={len(df):,}, clases={df.connected.value_counts().sort_index().to_dict()}')

# Desarrollo = todo 2021-2022. No hay una validacion fija de 7 observaciones.
dev = pd.concat([train, val], ignore_index=True)
dev['no_historical_path'] = dev['lenght_short_path'].isna().astype(int)
FEATURES = BASE_FEATURES + ['no_historical_path']

print(f'desarrollo train+val: n={len(dev):,}, clases={dev.connected.value_counts().sort_index().to_dict()}')
print('\nNaN en DESARROLLO por feature:')
print(dev[FEATURES].isna().sum().to_string())

# NaN de shortest path significa que no existe camino historico. Se codifica como 0
# y se acompana de no_historical_path=1 para no confundirlo con una distancia real.
X_dev_raw = dev[FEATURES].copy().fillna(0.0)
y_dev = dev['connected'].astype(int).to_numpy()

# Las variables constantes en TODO desarrollo no contienen informacion discriminativa.
dev_nunique = X_dev_raw.nunique(dropna=False)
CONSTANT_DEV = dev_nunique[dev_nunique <= 1].index.tolist()
MODEL_FEATURES = [f for f in FEATURES if f not in CONSTANT_DEV]

print('\nFeatures constantes en desarrollo (se excluyen):', CONSTANT_DEV)
print('Features candidatas:', MODEL_FEATURES)

# Comprobar que 5 folds son viables.
class_counts = pd.Series(y_dev).value_counts()
if class_counts.min() < N_SPLITS:
    raise ValueError(
        f'La clase minoritaria tiene {class_counts.min()} casos; no alcanza para {N_SPLITS} folds. '
        'Reduzca N_SPLITS.'
    )

# ============================================================
# MODELOS PARA ESTIMAR IMPORTANCIA
# ============================================================
# Se usan varios sesgos inductivos para que el ranking no dependa de un solo modelo.
# NO se afinan hiperparametros aqui; P5-1 hace esa etapa despues con el Top-K congelado.
models = {
    'Logistic Regression': LogisticRegression(max_iter=5000, random_state=SEED),
    'Random Forest': RandomForestClassifier(n_estimators=500, random_state=SEED, n_jobs=-1),
    'Gradient Boosting': GradientBoostingClassifier(random_state=SEED),
    'SVC RBF': SVC(kernel='rbf', C=1.0, gamma='scale', probability=True, random_state=SEED),
}

try:
    from xgboost import XGBClassifier
    models['XGBoost'] = XGBClassifier(
        n_estimators=300,
        max_depth=3,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.9,
        eval_metric='logloss',
        random_state=SEED,
        n_jobs=-1,
    )
    print('XGBoost disponible: se incluye.')
except Exception:
    print('XGBoost no disponible: P5 continua con sklearn.')

# ============================================================
# REPEATED STRATIFIED K-FOLD 5 x 20
# ============================================================
rkf = RepeatedStratifiedKFold(
    n_splits=N_SPLITS,
    n_repeats=N_REPEATS_CV,
    random_state=SEED,
)

rows_fold_metrics = []
rows_importance = []

for fold_id, (idx_tr, idx_va) in enumerate(rkf.split(X_dev_raw, y_dev), start=1):
    X_tr_raw = X_dev_raw.iloc[idx_tr][MODEL_FEATURES]
    X_va_raw = X_dev_raw.iloc[idx_va][MODEL_FEATURES]
    y_tr = y_dev[idx_tr]
    y_va = y_dev[idx_va]

    # Escalado aprendido exclusivamente con el fold de entrenamiento.
    scaler = StandardScaler()
    X_tr = scaler.fit_transform(X_tr_raw)
    X_va = scaler.transform(X_va_raw)

    for model_name, base_model in models.items():
        model = clone(base_model)
        model.fit(X_tr, y_tr)
        pred = model.predict(X_va)

        if hasattr(model, 'predict_proba'):
            score = model.predict_proba(X_va)[:, 1]
        elif hasattr(model, 'decision_function'):
            score = model.decision_function(X_va)
        else:
            score = pred.astype(float)

        rows_fold_metrics.append({
            'fold_id': fold_id,
            'model': model_name,
            'n_train_fold': len(idx_tr),
            'n_val_fold': len(idx_va),
            'accuracy': accuracy_score(y_va, pred),
            'f1': f1_score(y_va, pred, zero_division=0),
            'auc_roc': roc_auc_score(y_va, score),
            'auc_pr': average_precision_score(y_va, score),
        })

        # Importancia calculada SOLO sobre el fold held-out correspondiente.
        pi = permutation_importance(
            model,
            X_va,
            y_va,
            scoring='average_precision',
            n_repeats=N_REPEATS_IMPORTANCE,
            random_state=SEED + fold_id,
            n_jobs=-1,
        )

        order = np.argsort(pi.importances_mean)[::-1]
        top7_idx = set(order[:min(TOP_K, len(order))])

        for j, feature in enumerate(MODEL_FEATURES):
            rows_importance.append({
                'fold_id': fold_id,
                'model': model_name,
                'feature': feature,
                'importance_mean_aucpr': float(pi.importances_mean[j]),
                'importance_std_aucpr': float(pi.importances_std[j]),
                'positive_importance': int(pi.importances_mean[j] > 0),
                'in_fold_top7': int(j in top7_idx),
            })

fold_metrics = pd.DataFrame(rows_fold_metrics)
importance_long = pd.DataFrame(rows_importance)

# ============================================================
# RESUMEN DE METRICAS CV
# ============================================================
def q025(s):
    return s.quantile(0.025)

def q975(s):
    return s.quantile(0.975)

metrics_summary = (
    fold_metrics.groupby('model', as_index=False)
    .agg(
        folds=('fold_id', 'size'),
        accuracy_mean=('accuracy', 'mean'),
        accuracy_std=('accuracy', 'std'),
        f1_mean=('f1', 'mean'),
        f1_std=('f1', 'std'),
        auc_roc_mean=('auc_roc', 'mean'),
        auc_roc_std=('auc_roc', 'std'),
        auc_pr_mean=('auc_pr', 'mean'),
        auc_pr_std=('auc_pr', 'std'),
        auc_pr_median=('auc_pr', 'median'),
        auc_pr_q025=('auc_pr', q025),
        auc_pr_q975=('auc_pr', q975),
    )
    .sort_values('auc_pr_mean', ascending=False)
    .reset_index(drop=True)
)

print('\nDesempeno CV repetida en DESARROLLO (2021-2022; NO TEST):')
print(metrics_summary.to_string(index=False))

# ============================================================
# RANKING ROBUSTO DE FEATURES
# ============================================================
# Se resume sobre todos los pares modelo-fold. Se prioriza la importancia media;
# la frecuencia positiva y la frecuencia Top-7 sirven como estabilidad adicional.
importance_summary = (
    importance_long.groupby('feature', as_index=False)
    .agg(
        mean_importance=('importance_mean_aucpr', 'mean'),
        median_importance=('importance_mean_aucpr', 'median'),
        std_importance=('importance_mean_aucpr', 'std'),
        positive_fraction=('positive_importance', 'mean'),
        top7_frequency=('in_fold_top7', 'mean'),
        evaluations=('importance_mean_aucpr', 'size'),
    )
    .sort_values(
        ['mean_importance', 'positive_fraction', 'top7_frequency'],
        ascending=[False, False, False],
    )
    .reset_index(drop=True)
)
importance_summary['rank'] = np.arange(1, len(importance_summary) + 1)

selected_features = importance_summary.head(min(TOP_K, len(importance_summary)))['feature'].tolist()

print('\nRanking agregado de features por CV repetida:')
print(importance_summary.to_string(index=False))
print(f'\nTop-{len(selected_features)} congelado para P5-1:', selected_features)

# ============================================================
# SALIDAS
# ============================================================
OUT_FOLD_METRICS = BASE_DIR / 'P5_CV_fold_metrics.csv'
OUT_METRICS_SUMMARY = BASE_DIR / 'P5_CV_model_metrics_summary.csv'
OUT_LONG = BASE_DIR / 'P5_CV_feature_importance_by_fold.csv'
OUT_SUMMARY = BASE_DIR / 'P5_CV_feature_importance_summary.csv'
OUT_SELECTED = BASE_DIR / f'P5_selected_features_top{len(selected_features)}.txt'
OUT_AUDIT = BASE_DIR / 'P5_CV_audit.csv'

fold_metrics.to_csv(OUT_FOLD_METRICS, index=False)
metrics_summary.to_csv(OUT_METRICS_SUMMARY, index=False)
importance_long.to_csv(OUT_LONG, index=False)
importance_summary.to_csv(OUT_SUMMARY, index=False)
OUT_SELECTED.write_text('\n'.join(selected_features), encoding='utf-8')

pd.DataFrame({
    'item': [
        'n_train_original', 'n_val_original', 'n_development',
        'n_splits', 'n_repeats_cv', 'total_cv_folds',
        'n_features_original', 'n_features_model', 'constant_features_development',
        'preprocessing_scope', 'importance_scope', 'test_used_for_selection',
        'main_edge_threshold', 'note_independent_runs'
    ],
    'value': [
        len(train), len(val), len(dev),
        N_SPLITS, N_REPEATS_CV, N_SPLITS * N_REPEATS_CV,
        len(FEATURES), len(MODEL_FEATURES), ';'.join(CONSTANT_DEV),
        'within_each_training_fold_only',
        'held_out_cv_folds_within_2021_2022_development_only',
        'False',
        1,
        'Repeated CV stabilizes feature selection; it is not the inferential unit for the later multi-seed experiments.'
    ]
}).to_csv(OUT_AUDIT, index=False)

print('\nArchivos generados:')
for p in [OUT_FOLD_METRICS, OUT_METRICS_SUMMARY, OUT_LONG, OUT_SUMMARY, OUT_SELECTED, OUT_AUDIT]:
    print(' -', p.name)

# ============================================================
# EVALUACION FINAL OPCIONAL EN 2023-2024
# ============================================================
# Activar UNA VEZ que Top-K este congelado. El test no participa en seleccion.
if RUN_FINAL_TEST:
    print('\n*** EVALUACION FINAL EN TEST 2023-2024 ACTIVADA ***')
    test = load_split(TEST_FILE)
    missing = required.difference(test.columns)
    if missing:
        raise ValueError(f'test: faltan columnas: {sorted(missing)}')
    test['no_historical_path'] = test['lenght_short_path'].isna().astype(int)

    X_train_final_raw = X_dev_raw[selected_features].copy()
    X_test_final_raw = test[selected_features].copy().fillna(0.0)
    y_test = test['connected'].astype(int).to_numpy()

    scaler_final = StandardScaler()
    X_train_final = scaler_final.fit_transform(X_train_final_raw)
    X_test_final = scaler_final.transform(X_test_final_raw)

    final_rows = []
    for model_name, base_model in models.items():
        model = clone(base_model)
        model.fit(X_train_final, y_dev)
        pred = model.predict(X_test_final)
        if hasattr(model, 'predict_proba'):
            score = model.predict_proba(X_test_final)[:, 1]
        elif hasattr(model, 'decision_function'):
            score = model.decision_function(X_test_final)
        else:
            score = pred.astype(float)

        final_rows.append({
            'model': model_name,
            'accuracy_test': accuracy_score(y_test, pred),
            'f1_test': f1_score(y_test, pred, zero_division=0),
            'auc_roc_test': roc_auc_score(y_test, score),
            'auc_pr_test': average_precision_score(y_test, score),
        })

    final_df = pd.DataFrame(final_rows).sort_values('auc_pr_test', ascending=False)
    print(final_df.to_string(index=False))
    final_df.to_csv(BASE_DIR / 'P5_FINAL_test_metrics_repeatedCV.csv', index=False)


train: n=27, clases={0: 13, 1: 14}
val: n=7, clases={0: 4, 1: 3}
desarrollo train+val: n=34, clases={0: 17, 1: 17}

NaN en DESARROLLO por feature:
sum_of_papers               0
sum_of_neighbors            0
log_secundary_neighbors     0
sum_of_keywords             0
keywords_match              0
sum_of_areas                0
lenght_short_path          17
clustering_index_sum        0
jaccard_coefficient         0
resource_allocation         0
adamic_adar_index           0
preferential_attachment     0
community_cn                0
community_ra                0
no_historical_path          0

Features constantes en desarrollo (se excluyen): []
Features candidatas: ['sum_of_papers', 'sum_of_neighbors', 'log_secundary_neighbors', 'sum_of_keywords', 'keywords_match', 'sum_of_areas', 'lenght_short_path', 'clustering_index_sum', 'jaccard_coefficient', 'resource_allocation', 'adamic_adar_index', 'preferential_attachment', 'community_cn', 'community_ra', 'no_historical_path']
XGBoost no disponi